In [2]:
!pip install music21 tensorflow

In [3]:
#Download and prepare MAESTRO dataset
!wget -q https://storage.googleapis.com/magentadata/datasets/maestro/v2.0.0/maestro-v2.0.0-midi.zip
!unzip -oq maestro-v2.0.0-midi.zip

import os
import shutil

os.makedirs("midi_dataset", exist_ok=True)

for root, _, files in os.walk("maestro-v2.0.0"):
    for file in files:
        if file.endswith(".midi") or file.endswith(".mid"):  # FIX: ловим оба расширения
            shutil.copy(os.path.join(root, file), "midi_dataset")

print("DONE. Files collected:", len(os.listdir("midi_dataset")))

DONE. Files collected: 1282


In [4]:
#  Parse notes from MIDI files
from music21 import converter, note, chord

MAX_NOTES = 100000

def get_notes():
    notes = []
    files = os.listdir("midi_dataset")[:150]

    for i, file in enumerate(files):
        try:
            midi = converter.parse(f"midi_dataset/{file}")
            elements = midi.flat.notes

            for el in elements:
                if len(notes) >= MAX_NOTES:  # FIX: >= вместо >
                    return notes

                if isinstance(el, note.Note):
                    notes.append(f"{el.pitch}_{round(el.duration.quarterLength, 1)}")
                elif isinstance(el, chord.Chord):
                    chord_notes = '.'.join(str(n) for n in el.normalOrder)
                    notes.append(f"{chord_notes}_{round(el.duration.quarterLength, 1)}")

        except Exception as e:
            print(f"  Skipped {file}: {e}")  # FIX: не глотаем ошибки молча
            continue

    return notes

notes = get_notes()
print("Total notes parsed:", len(notes))

/usr/local/lib/python3.12/dist-packages/music21/stream/base.py:3675: Music21DeprecationWarning: .flat is deprecated.  Call .flatten() instead
  return self.iter().getElementsByClass(classFilterList)


Total notes parsed: 100000


In [5]:
#Build vocabulary
import numpy as np

pitchnames  = sorted(set(notes))
n_vocab     = len(pitchnames)
note_to_int = {n: i for i, n in enumerate(pitchnames)}
int_to_note = {i: n for i, n in enumerate(pitchnames)}

print("Vocabulary size:", n_vocab)

Vocabulary size: 6171


In [6]:
# Data generator
sequence_length = 50
batch_size      = 32

def data_generator(notes, batch_size, note_to_int):

    while True:
        X, y = [], []
        for _ in range(batch_size):
            start   = np.random.randint(0, len(notes) - sequence_length)
            seq_in  = notes[start : start + sequence_length]
            seq_out = notes[start + sequence_length]
            X.append([note_to_int[n] for n in seq_in])
            y.append(note_to_int[seq_out])
        yield np.array(X), np.array(y)

# Build a small val split for EarlyStopping
val_size  = 1000
val_start = np.random.randint(0, len(notes) - sequence_length - val_size)
val_X, val_y = [], []
for i in range(val_start, val_start + val_size):
    val_X.append([note_to_int[n] for n in notes[i : i + sequence_length]])
    val_y.append(note_to_int[notes[i + sequence_length]])
val_X = np.array(val_X)
val_y = np.array(val_y)

print("Validation set size:", len(val_y))

Validation set size: 1000


In [7]:
# Build Transformer model
import tensorflow as tf
from tensorflow.keras import layers

def positional_encoding(seq_len, d_model):

    positions = np.arange(seq_len)[:, np.newaxis]
    dims      = np.arange(d_model)[np.newaxis, :]
    angles    = positions / np.power(10000, (2 * (dims // 2)) / d_model)
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles[np.newaxis, :, :], dtype=tf.float32)

def transformer_block(inputs, head_size, num_heads, ff_dim, dropout=0.1):
    x = layers.MultiHeadAttention(num_heads=num_heads, key_dim=head_size)(inputs, inputs)
    x = layers.Dropout(dropout)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + inputs)

    # Feed-Forward
    ffn = layers.Dense(ff_dim, activation="relu")(x)
    ffn = layers.Dense(inputs.shape[-1])(ffn)
    ffn = layers.Dropout(dropout)(ffn)
    return layers.LayerNormalization(epsilon=1e-6)(x + ffn)

d_model = 128

inputs    = layers.Input(shape=(sequence_length,))
x         = layers.Embedding(n_vocab, d_model)(inputs)

pe        = positional_encoding(sequence_length, d_model)
x         = x + pe

x = transformer_block(x, head_size=64, num_heads=4, ff_dim=256)
x = transformer_block(x, head_size=64, num_heads=4, ff_dim=256)
x = transformer_block(x, head_size=64, num_heads=4, ff_dim=256)

x       = layers.GlobalAveragePooling1D()(x)
x       = layers.Dense(256, activation="relu")(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(n_vocab, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)
model.compile(
    loss      = "sparse_categorical_crossentropy",
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0003),
    metrics   = ["accuracy"]
)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 50, 128)   │    789,888 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 50, 128)   │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 50, 128)   │    131,968 │ add[0][0],        │
│ (MultiHeadAttentio… │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 50, 128)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 50, 128)   │          0 │ dropout_1[0][0],  │
│                     │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 50, 128)   │        256 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 50, 256)   │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 50, 128)   │     32,896 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 50, 128)   │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 50, 128)   │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 128)   │        256 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 50, 128)   │    131,968 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 50, 128)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 50, 128)   │          0 │ dropout_4[0][0],  │
│                     │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 128)   │        256 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 50, 256)   │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 50, 128)   │     32,896 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 50, 128)   │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 50, 128)   │          0 │ layer_normalizat

 Total params: 3,004,059 (11.46 MB)

 Trainable params: 3,004,059 (11.46 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
#  Train
gen = data_generator(notes, batch_size, note_to_int)

callbacks = [

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),

    tf.keras.callbacks.ModelCheckpoint(
        "best_music_model.keras", monitor="val_loss", save_best_only=True
    ),
]

model.fit(
    gen,
    steps_per_epoch = 800,
    epochs          = 30,
    validation_data = (val_X, val_y),
    callbacks       = callbacks,
)

np.save("note_to_int.npy", note_to_int)
np.save("int_to_note.npy", int_to_note)
print("Model and vocab saved.")

Epoch 1/30
800/800 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - accuracy: 0.0047 - loss: 7.3025 - val_accuracy: 0.0020 - val_loss: 7.3514
Epoch 2/30
800/800 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.0046 - loss: 7.2570 - val_accuracy: 0.0050 - val_loss: 7.3830
Epoch 3/30
800/800 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.0052 - loss: 7.2110 - val_accuracy: 0.0020 - val_loss: 7.3738
Epoch 4/30
800/800 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.0055 - loss: 7.1919 - val_accuracy: 0.0050 - val_loss: 7.3804
Epoch 5/30
800/800 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.0056 - loss: 7.1887 - val_accuracy: 0.0030 - val_loss: 7.3175
Epoch 6/30
800/800 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.0063 - loss: 7.1672 - val_accuracy: 0.0040 - val_loss: 7.3101
Epoch 7/30
800/800 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.0068 - loss: 7.1562 - val_accuracy: 0.0040 - val_loss: 7.2766
Epoch 8/30
800/800 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.0063 - loss: 7.1560 - val_accurac

In [26]:
# Generate melody
import random
from music21 import stream

TEMPERATURE   = 1.0
MELODY_LENGTH = 400
SPEED         = 0.9
MIN_DURATION  = 0.5
MAX_DURATION  = 2.0

def sample_with_temperature(preds, temperature=0.7):
    preds  = np.log(preds + 1e-9) / temperature
    preds  = np.exp(preds)
    preds /= np.sum(preds)
    return np.random.choice(len(preds), p=preds)

start   = random.randint(0, len(notes) - sequence_length)
pattern = [note_to_int[n] for n in notes[start : start + sequence_length]]
output  = []

for _ in range(MELODY_LENGTH):
    inp        = np.array(pattern).reshape(1, -1)
    prediction = model.predict(inp, verbose=0)[0]
    index      = sample_with_temperature(prediction, TEMPERATURE)
    output.append(int_to_note[index])
    pattern    = pattern[1:] + [index]


In [27]:
from fractions import Fraction
OUTPUT_FILE = "ai_project_melody.mid"

output_notes_final = []
offset = 0

for item in output:
    try:
        parts = item.split("_")
        if len(parts) < 2:
            continue

        pitch    = parts[0]
        duration = float(Fraction(parts[1]))
        duration = max(MIN_DURATION, min(duration, MAX_DURATION)) / SPEED
        velocity = int(np.random.randint(60, 100))

        if pitch.lower() == "rest":
            new_note = note.Rest()

        elif "." in pitch:

            nums      = [p for p in pitch.split(".") if p.isdigit()]
            if not nums:
                continue
            chord_notes = [note.Note(int(n)) for n in nums]
            new_note    = chord.Chord(chord_notes)
            new_note.volume.velocity = velocity

        else:
            new_note = note.Note(int(pitch)) if pitch.isdigit() else note.Note(pitch)
            new_note.volume.velocity = velocity

        new_note.duration.quarterLength = duration
        new_note.offset = offset
        output_notes_final.append(new_note)
        offset += duration

    except Exception as e:
        print(f"  Skipped token '{item}': {e}")  # FIX: не глотаем ошибки
        continue

midi_stream = stream.Stream(output_notes_final)
file_path   = midi_stream.write("midi", fp=OUTPUT_FILE)
print("Saved:", file_path)

Saved: ai_project_melody.mid


In [28]:
from google.colab import files
files.download(OUTPUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
!git add .
!git commit -m "Update notebook"
!git push

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
remote: Invalid username or token. Password authentication is not supported for Git operations.
fatal: Authentication failed for 'https://github.com/Tigran112/Music-Generation.git/'
